In [1]:
!pip install -q transformers accelerate

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("Loading model onto the GPU (first run downloads ~15 GB, a few minutes)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",   # puts it on the Colab GPU automatically
)
print("Model ready.")

Loading model onto the GPU (first run downloads ~15 GB, a few minutes)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model ready.


In [3]:
# ============================================================
#  SYNTHETIC DATA MODE  —  no Vortexa key, no licensed data.
#  Run this cell INSTEAD of the real Cell 4 for a public demo.
#  Every number below is randomly generated, NOT real Vortexa data.
# ============================================================
import json, random
from datetime import datetime, timedelta

print("*** SYNTHETIC MODE: all figures are fake, for demonstration only ***")

VESSEL_CLASSES = [
    "handysize", "handymax", "panamax", "aframax", "suezmax", "vlcc",
    "mr1", "mr2", "lr1", "lr2", "lr3",
]

# realistic-looking destinations/origins so summaries read naturally
_PLACES = ["Mexico", "United States", "Brazil", "Colombia", "Guatemala",
           "Netherlands", "Singapore", "China", "Japan", "South Korea",
           "India", "Nigeria", "Chile", "Peru", "Australia"]

# --- tool definitions: IDENTICAL to the real ones, so routing is unchanged ---
QUERY_TOOL = {
    "type": "function",
    "function": {
        "name": "query_cargo_flows",
        "description": (
            "Query total cargo movements (tanker flows) for ONE time period — a "
            "single total. Use for 'how much moved from A to B in month X'. "
            "'loading_state' = exports, 'unloading_state' = imports. Resolve "
            "relative dates to YYYY-MM-DD using today's date."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "activity": {"type": "string",
                             "enum": ["loading_state", "unloading_state", "any_activity"]},
                "origins": {"type": "array", "items": {"type": "string"}},
                "destinations": {"type": "array", "items": {"type": "string"}},
                "products": {"type": "array", "items": {"type": "string"}},
                "vessel_classes": {"type": "array",
                                   "items": {"type": "string", "enum": VESSEL_CLASSES}},
                "date_from": {"type": "string", "description": "YYYY-MM-DD"},
                "date_to": {"type": "string", "description": "YYYY-MM-DD"},
                "group_by": {"type": "string",
                             "enum": ["destination_country", "origin_country",
                                      "product", "vessel_class", "none"]},
            },
            "required": ["activity", "date_from", "date_to"],
        },
    },
}

TIMESERIES_TOOL = {
    "type": "function",
    "function": {
        "name": "query_cargo_timeseries",
        "description": (
            "Get a TREND of cargo volumes OVER TIME in monthly or daily buckets. "
            "Use for 'over the last year', 'month by month', 'how has X changed'. "
            "Returns a time series, not a single total. 'loading_state' = exports, "
            "'unloading_state' = imports. Resolve relative dates to YYYY-MM-DD."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "activity": {"type": "string",
                             "enum": ["loading_state", "unloading_state", "any_activity"]},
                "origins": {"type": "array", "items": {"type": "string"}},
                "destinations": {"type": "array", "items": {"type": "string"}},
                "products": {"type": "array", "items": {"type": "string"}},
                "vessel_classes": {"type": "array",
                                   "items": {"type": "string", "enum": VESSEL_CLASSES}},
                "date_from": {"type": "string", "description": "YYYY-MM-DD"},
                "date_to": {"type": "string", "description": "YYYY-MM-DD"},
                "frequency": {"type": "string", "enum": ["day", "month"],
                              "description": "Bucket size; default month."},
            },
            "required": ["activity", "date_from", "date_to"],
        },
    },
}


def _seeded(params):
    """Deterministic per-question randomness, so the same query gives the same
    fake answer — feels less obviously random in a demo."""
    key = json.dumps(params, sort_keys=True)
    random.seed(hash(key) % (2**32))


def run_vortexa_query(params):
    _seeded(params)
    activity = params.get("activity", "loading_state")
    print(f"  [synthetic] flows {activity} "
          f"{params['date_from']} -> {params['date_to']} ...")

    rows = random.randint(120, 900)
    total = rows * random.randint(250_000, 420_000)
    agg = {
        "rows": rows,
        "total_quantity_bbl": float(total),
        "unique_grades": random.randint(2, 6),
        "_note": "SYNTHETIC DATA — not real Vortexa figures",
    }
    group_by = params.get("group_by", "none")
    if group_by in ("destination_country", "origin_country"):
        picks = random.sample(_PLACES, random.randint(3, 5))
        weights = sorted([random.random() for _ in picks], reverse=True)
        s = sum(weights)
        agg["top_" + group_by] = {
            p: round(total * w / s) for p, w in zip(picks, weights)
        }
    return agg


def run_timeseries_query(params):
    _seeded(params)
    activity = params.get("activity", "loading_state")
    freq = params.get("frequency", "month")
    start = datetime.strptime(params["date_from"], "%Y-%m-%d")
    end = datetime.strptime(params["date_to"], "%Y-%m-%d")
    print(f"  [synthetic] timeseries {activity} {freq} "
          f"{params['date_from']} -> {params['date_to']} ...")

    # build monthly buckets between the dates
    series = {}
    cur = start
    base = random.randint(30_000_000, 40_000_000)
    while cur <= end:
        # gentle seasonal wobble around the base
        val = base * (1 + 0.12 * random.uniform(-1, 1))
        series[cur.strftime("%Y-%m")] = round(val)
        # step forward ~1 month
        cur = (cur.replace(day=1) + timedelta(days=32)).replace(day=1)

    values = list(series.values())
    return {
        "frequency": freq,
        "buckets": len(series),
        "total_bbl": float(sum(values)),
        "first_bucket": {"date": list(series)[0], "value": values[0]} if values else None,
        "last_bucket": {"date": list(series)[-1], "value": values[-1]} if values else None,
        "series": series,
        "_note": "SYNTHETIC DATA — not real Vortexa figures",
    }

*** SYNTHETIC MODE: all figures are fake, for demonstration only ***


In [4]:
import re

def generate(messages):
    text = tokenizer.apply_chat_template(
        messages, tools=[QUERY_TOOL, TIMESERIES_TOOL],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                             skip_special_tokens=True)
    return reply


def extract_tool_calls(reply):
    calls = []
    for m in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", reply, re.DOTALL):
        try:
            calls.append(json.loads(m))
        except json.JSONDecodeError:
            pass
    return calls


def chat_once(user_input, messages):
    messages.append({"role": "user", "content": user_input})
    reply = generate(messages)
    calls = extract_tool_calls(reply)

    if calls:
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": c["name"],
             "arguments": c.get("arguments", {})}} for c in calls]})
        for c in calls:
            params = c.get("arguments", {})
            print(f"  [model->query] {c['name']} {json.dumps(params)}")
            try:
                if c["name"] == "query_cargo_flows":
                    result = run_vortexa_query(params)
                elif c["name"] == "query_cargo_timeseries":
                    result = run_timeseries_query(params)
                else:
                    result = {"error": f"unknown tool {c['name']}"}
            except Exception as e:
                result = {"error": str(e)}
            messages.append({"role": "tool", "name": c["name"],
                             "content": json.dumps(result, default=str)})
        reply = generate(messages)

    messages.append({"role": "assistant", "content": reply})
    return reply

In [5]:
today = datetime.now().strftime("%Y-%m-%d")
system_prompt = (
    "You are a maritime freight analyst assistant with access to Vortexa "
    "cargo-flow data via the query_cargo_flows tool. "
    f"Today's date is {today}. Call the tool with structured parameters, "
    "resolving relative dates yourself. After the tool returns results, write a "
    "concise analyst summary: headline number in barrels (bbl), then the notable "
    "breakdown, then one caveat if relevant. Only use figures the tool returns."
)
messages = [{"role": "system", "content": system_prompt}]



In [6]:
print("Maritime cargo-flow chatbot. Type 'quit' to stop.\n")
while True:
    question = input("You: ")
    if question.strip().lower() in {"quit", "exit", "q"}:
        print("Done.")
        break
    if not question.strip():
        continue
    answer = chat_once(question, messages)
    print("\nBot:", answer, "\n")

Maritime cargo-flow chatbot. Type 'quit' to stop.

You: MR Tanker gasoline exports from the US Gulf in June 2026, grouped by destination
  [model->query] query_cargo_flows {"activity": "loading_state", "origins": ["US Gulf"], "destinations": [], "products": ["Gasoline"], "vessel_classes": ["mr1", "mr2"], "date_from": "2026-06-01", "date_to": "2026-06-30", "group_by": "destination_country"}
  [synthetic] flows loading_state 2026-06-01 -> 2026-06-30 ...

Bot: Headline: 227,773,043 barrels of MR tanker gasoline were exported from the US Gulf in June 2026.

Notable Breakdown: The top destinations were Mexico (124,569,398 bbl), Brazil (54,155,520 bbl), and Singapore (49,048,125 bbl).

Caveat: These figures are synthetic data and not actual Vortexa figures. 



KeyboardInterrupt: Interrupted by user